In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# §6.26 The Density Matrix, Mixed States, and Decoherence

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.26",
    title="The Density Matrix, Mixed States, and Decoherence",
    blurb="What is the state of half an entangled pair? No single wavefunction "
    "will do. The density matrix is the object that describes it — and with it, "
    "the difference between a genuine superposition and a mere statistical "
    "mixture, the entanglement of a subsystem measured as an entropy, and the way "
    "a quantum system loses its coherence to the environment and starts to look "
    "classical. It is also the state on which quantum statistical mechanics will "
    "later be built.",
    difficulty="advanced",
    estimate="170–210 min",
)

## Notebook overview

A puzzle has run through this movement: what *is* the state of one half of an
entangled pair? The Bell state ([§6.8](bloch-sphere-entanglement.ipynb)) and the Bell inequality ([§6.25](bell-inequality.ipynb)) kept showing
single-particle marginals that looked completely random, yet no state vector for
a single particle reproduces that. A state vector $|\psi\rangle$ describes a
system we know as well as quantum mechanics allows — a **pure** state — but two
situations escape it: a **statistical mixture** (we are classically uncertain
which pure state we have, like an unpolarized beam) and a **subsystem** of an
entangled whole (where no state vector for the part exists at all). The **density
operator** $\rho$ handles both.

For a pure state $\rho=|\psi\rangle\langle\psi|$; for a mixture $\rho=\sum_i p_i
|\psi_i\rangle\langle\psi_i|$. It is Hermitian, positive, and unit-trace, and
every prediction follows from it, $\langle A\rangle=\mathrm{Tr}(\rho A)$.
Crucially $\rho$ distinguishes a coherent **superposition** from a **mixture** —
the superposition carries nonzero off-diagonal *coherences* and a definite value
in a rotated basis, the mixture does not — a distinction a bare list of
probabilities erases. The **purity** $\mathrm{Tr}(\rho^2)$ diagnoses which. For a
composite system the state of one part is the **reduced density matrix**
$\rho_A=\mathrm{Tr}_B\rho$, and here the deep fact surfaces: a *pure* entangled
whole has *mixed* parts. The reduced state of a Bell pair is maximally mixed
(purity $\tfrac12$, von Neumann entropy $1$ bit) — which is exactly *why* the
marginals of [§6.8](bloch-sphere-entanglement.ipynb) and [§6.25](bell-inequality.ipynb) were random. That entropy measures the entanglement.
Finally **decoherence**: when a superposition entangles with an environment we
then ignore, its coherences decay and it becomes a classical mixture — the
mechanism of the quantum-to-classical transition, and the practical adversary of
every qubit.

> **Scope boundary.** We develop $\rho$ for **entanglement, mixed states, and
> decoherence**. The same object also carries a *thermal* weight, $\rho=e^{-\beta
> H}/Z$, and is the starting point of quantum statistical mechanics — but the
> thermal state, partition functions, and the Fermi–Dirac / Bose–Einstein
> distributions are **deferred to Volume VII**. Here $\rho$ is the language of
> entanglement. Lindblad / open-system dynamics and quantum channels are named as
> horizons, not developed.

> **Method specificity.** $\rho=|\psi\rangle\langle\psi|$ via `numpy.outer`;
> $\langle A\rangle=\mathrm{Tr}(\rho A)$ via `numpy.trace`; the partial trace via
> `numpy.reshape` + `numpy.trace` over the traced indices; the von Neumann entropy
> from `numpy.linalg.eigvalsh`; and `scipy.linalg.expm` for the von Neumann
> evolution. Entropy is in **bits** ($\log_2$).

## Theory in brief

### Why a state vector is not enough

$|\psi\rangle$ describes a pure state — maximal knowledge. But an unpolarized beam
(a classical 50/50 mixture of up and down) and one half of an entangled pair
cannot be written as any single $|\psi\rangle$. We need an object for mixtures and
subsystems.

```{math}
:label: eq-why-rho
\text{pure: } |\psi\rangle \qquad\text{vs}\qquad
\text{mixture / subsystem: no single } |\psi\rangle \text{ exists.}
```

### The density operator

For a pure state $\rho=|\psi\rangle\langle\psi|$; for a mixture of pure states
$|\psi_i\rangle$ with classical probabilities $p_i$,

```{math}
:label: eq-density-operator
\rho = \sum_i p_i\,|\psi_i\rangle\langle\psi_i| , \qquad
\rho=\rho^\dagger,\quad \rho\succeq0,\quad \mathrm{Tr}\,\rho=1 ,
```

and every prediction is $\langle A\rangle=\mathrm{Tr}(\rho A)$ (equal to
$\langle\psi|A|\psi\rangle$ for a pure state); the diagonal entries are outcome
probabilities.

### Superposition versus mixture

The essential lesson. The pure superposition $(|0\rangle+|1\rangle)/\sqrt2$ and
the mixture $\tfrac12|0\rangle\langle0|+\tfrac12|1\rangle\langle1|$ have the *same*
diagonal (50/50 in the computational basis) but *different* off-diagonals:

```{math}
:label: eq-superposition-mixture
\rho_{\text{sup}}=\tfrac12\begin{pmatrix}1&1\\1&1\end{pmatrix},\qquad
\rho_{\text{mix}}=\tfrac12\begin{pmatrix}1&0\\0&1\end{pmatrix}.
```

The superposition's off-diagonal **coherences** make it give a definite result in
a rotated basis ($\langle\sigma_x\rangle=1$); the mixture stays random
($\langle\sigma_x\rangle=0$). They are physically distinct states.

### Purity

The **purity** $\mathrm{Tr}(\rho^2)$ is $1$ iff $\rho$ is pure, and smaller for
mixed states, down to $1/d$ for the maximally mixed state.

```{math}
:label: eq-purity
\mathrm{Tr}(\rho^2) = \frac{1+r^2}{2}\quad(\text{qubit, Bloch length }r):
\ 1 \text{ on the Bloch surface},\ \tfrac12 \text{ at the center.}
```

The Bloch sphere of [§6.8](bloch-sphere-entanglement.ipynb), now filled in: pure states on the surface, mixed states
in the interior.

### The reduced density matrix and the partial trace

For a composite system $AB$, the state of $A$ alone is the **reduced density
matrix** obtained by the **partial trace** over $B$,

```{math}
:label: eq-partial-trace
\rho_A = \mathrm{Tr}_B\,\rho_{AB} = \sum_k \langle k|_B\,\rho_{AB}\,|k\rangle_B ,
```

computed by reshaping to $(d_A,d_B,d_A,d_B)$ and tracing the $B$ indices. It gives
the correct statistics for any measurement on $A$ alone.

### Entanglement makes parts mixed

The deep result. For a Bell state the *whole* is pure but each *part* is maximally
mixed:

```{math}
:label: eq-entanglement-mixed
|\Phi\rangle=\tfrac1{\sqrt2}(|00\rangle+|11\rangle)\ \text{pure},\qquad
\rho_A=\tfrac12 I,\quad S(\rho_A)=-\!\sum_k\lambda_k\log_2\lambda_k = 1\ \text{bit.}
```

A pure entangled state has mixed subsystems — exactly *why* the single-particle
marginals of [§6.8](bloch-sphere-entanglement.ipynb) and [§6.25](bell-inequality.ipynb) are random. The **von Neumann entropy** $S(\rho)$ is
$0$ for a pure state and $1$ bit for a maximally mixed qubit; for a pure bipartite
state $S(\rho_A)$ is the **entanglement entropy**.

### Decoherence

How the classical world emerges. Let a superposition entangle with an environment
that records which state it is in, $(|0\rangle|E_0\rangle+|1\rangle|E_1\rangle)/
\sqrt2$; tracing out the environment leaves a system coherence proportional to the
overlap $\langle E_0|E_1\rangle$:

```{math}
:label: eq-decoherence
\rho_S = \tfrac12\begin{pmatrix}1&\langle E_1|E_0\rangle\\ \langle E_0|E_1\rangle&1
\end{pmatrix} \xrightarrow{\ \langle E_0|E_1\rangle\to0\ } \tfrac12 I .
```

As the environment distinguishes the states, the coherences decay and the pure
superposition becomes a classical mixture (purity $1\to\tfrac12$) — why
macroscopic superpositions are never seen, and why qubits must be isolated.

### Evolution of the density matrix

For a closed system, $\rho(t)=U(t)\rho(0)U^\dagger(t)$, equivalently the **von
Neumann equation**

```{math}
:label: eq-von-neumann
i\hbar\,\frac{\mathrm d\rho}{\mathrm dt} = [H,\rho] ,
```

the density-matrix form of the Schrödinger equation. Under unitary evolution
**purity is conserved** — mixing never arises from closed dynamics, only from
tracing out an environment (decoherence) or classical ignorance.

- Reference: Sakurai & Napolitano (§3.4); Nielsen & Chuang {cite}`nielsen_chuang`
  (density matrices, partial trace, von Neumann entropy); Griffiths
  {cite}`griffiths_qm` (mixed states). Cross-reference [§6.8](bloch-sphere-entanglement.ipynb) (the Bloch sphere, now
  filled in; the Bell reduced state), [§6.25](bell-inequality.ipynb) (the random Bell marginals, now
  explained), [§6.5](postulates.ipynb) (the Born rule), [§6.7](time-evolution.ipynb) (unitary evolution, now for $\rho$), and
  forward to [§6.27](quantum-information.ipynb) (quantum information) and Volume VII (the thermal density matrix
  — the deferred home). Named as horizons: Lindblad / open-system dynamics,
  quantum channels, the thermal state.

---
## Setup

We work with qubits (subsystem dimension 2), entropy in **bits** ($\log_2$).
Conventions: for a two-qubit state the first factor is $A$; the partial trace
reshapes $\rho$ to $(d_A,d_B,d_A,d_B)$ and traces the chosen pair of indices. The
reusable core:

- `density_matrix(spec)`: $\rho=|\psi\rangle\langle\psi|$ (`numpy.outer`) from a
  ket, or $\sum_i p_i|\psi_i\rangle\langle\psi_i|$ from a list of `(p, ket)`.
- `expectation(rho, A)`: $\langle A\rangle=\mathrm{Tr}(\rho A)$ (`numpy.trace`).
- `partial_trace(rho, keep, dims)`: the reduced density matrix
  {eq}`eq-partial-trace` (`numpy.reshape` + `numpy.trace`).
- `purity(rho)`: $\mathrm{Tr}(\rho^2)$.
- `von_neumann_entropy(rho)`: $-\sum_k\lambda_k\log_2\lambda_k$ from
  `numpy.linalg.eigvalsh`.
- `evolve_density(rho, H, t)`: $U\rho U^\dagger$ with $U=e^{-iHt/\hbar}$
  (`scipy.linalg.expm`).

In [ ]:
import numpy as np
from scipy.linalg import expm
import matplotlib.pyplot as plt

from ecp import draw, validate

HBAR = 1.0
KET0 = np.array([1.0, 0.0], dtype=complex)
KET1 = np.array([0.0, 1.0], dtype=complex)
SIGMA_X = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
SIGMA_Y = np.array([[0.0, -1j], [1j, 0.0]], dtype=complex)
SIGMA_Z = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)


def density_matrix(spec):
    r"""Density operator from a pure ket or a classical mixture.

    A pure state gives $\rho=|\psi\rangle\langle\psi|$ via `numpy.outer`; a list of
    ``(p, ket)`` pairs gives the mixture $\sum_i p_i|\psi_i\rangle\langle\psi_i|$
    {eq}`eq-density-operator`.

    Parameters
    ----------
    spec : numpy.ndarray or list of (float, numpy.ndarray)
        Either a state vector, or ``[(p_i, psi_i), ...]`` with probabilities.

    Returns
    -------
    numpy.ndarray
        The density matrix.
    """
    if isinstance(spec, list):
        return sum(p * np.outer(psi, psi.conj()) for p, psi in spec)
    return np.outer(spec, spec.conj())


def expectation(rho, A):
    r"""Expectation value $\langle A\rangle=\mathrm{Tr}(\rho A)$ (`numpy.trace`)."""
    return float(np.real(np.trace(rho @ A)))


def partial_trace(rho, keep, dims):
    r"""Reduced density matrix by the partial trace {eq}`eq-partial-trace`.

    Reshapes $\rho$ to ``(dA, dB, dA, dB)`` and traces out the subsystem *not* in
    ``keep`` with `numpy.trace`.

    Parameters
    ----------
    rho : numpy.ndarray
        The joint density matrix.
    keep : int
        Which subsystem to keep (0 = A, 1 = B).
    dims : tuple of int
        ``(dA, dB)`` subsystem dimensions.

    Returns
    -------
    numpy.ndarray
        The reduced density matrix of the kept subsystem.
    """
    dA, dB = dims
    r = rho.reshape(dA, dB, dA, dB)
    if keep == 0:
        return np.trace(r, axis1=1, axis2=3)  # trace out B
    return np.trace(r, axis1=0, axis2=2)  # trace out A


def purity(rho):
    r"""Purity $\mathrm{Tr}(\rho^2)$ {eq}`eq-purity`: 1 for pure, $1/d$ for maximally mixed."""
    return float(np.real(np.trace(rho @ rho)))


def von_neumann_entropy(rho):
    r"""Von Neumann entropy $S=-\sum_k\lambda_k\log_2\lambda_k$ in bits.

    The eigenvalues come from `numpy.linalg.eigvalsh`; zero (and tiny negative,
    round-off) eigenvalues are dropped, since $0\log0=0$.

    Parameters
    ----------
    rho : numpy.ndarray
        A density matrix.

    Returns
    -------
    float
        The entropy in bits.
    """
    lam = np.linalg.eigvalsh(rho)
    lam = lam[lam > 1e-12]
    return float(-np.sum(lam * np.log2(lam)))


def evolve_density(rho, H, t):
    r"""Unitary (von Neumann) evolution $U\rho U^\dagger$, $U=e^{-iHt/\hbar}$ (`scipy.linalg.expm`)."""
    U = expm(-1j * H * t / HBAR)
    return U @ rho @ U.conj().T

## Exercise 1 — The density operator and its properties

Build the density matrix for a pure state and for a mixture, and verify its defining properties:
Hermitian, positive, unit-trace, and $\langle A\rangle=\mathrm{Tr}(\rho A)$.

1. Form $\rho=|\psi\rangle\langle\psi|$ (`numpy.outer`) for a pure state and
   $\rho=\sum_i p_i|\psi_i\rangle\langle\psi_i|$ for a mixture.
2. Check Hermiticity, positivity (eigenvalues $\ge0$), and $\mathrm{Tr}\,\rho=1$.
3. Confirm $\langle A\rangle=\mathrm{Tr}(\rho A)$ equals $\langle\psi|A|\psi\rangle$
   for the pure case (`numpy.trace`).
4. Note the diagonal entries are outcome probabilities.

Cite {eq}`eq-density-operator`.

In [ ]:
# (solution hidden on the public site)


### Validation 1

The density operator must be Hermitian, positive, and unit-trace, and the trace
formula must reproduce the state-vector expectation: $\mathrm{Tr}(\rho A)=\langle
\psi|A|\psi\rangle$.

In [ ]:
validate.check(
    np.allclose(rho_pure, rho_pure.conj().T)
    and np.all(np.linalg.eigvalsh(rho_pure) > -1e-12)
    and np.isclose(np.trace(rho_pure).real, 1.0),
    "the density operator is Hermitian, positive, and unit-trace",
)
validate.close(tr_form, bra_ket, "expectation values are ⟨A⟩ = Tr(ρA)", rtol=1e-9)

## Exercise 2 — Superposition versus mixture

Show that a coherent superposition and a statistical mixture with the *same* measurement
probabilities in one basis are nonetheless different states — the difference lives in the
off-diagonal coherences.

1. Build $\rho$ for the superposition $(|0\rangle+|1\rangle)/\sqrt2$ and for the
   50/50 mixture $\tfrac12|0\rangle\langle0|+\tfrac12|1\rangle\langle1|$.
2. Compare their matrices: same diagonal, different off-diagonal.
3. Compute the purity of each ($1$ vs $\tfrac12$).
4. Measure $\langle\sigma_x\rangle$: definite ($1$) for the superposition, random
   ($0$) for the mixture — a measurable difference.

Cite {eq}`eq-superposition-mixture`, {eq}`eq-purity`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

The superposition is pure (purity $1$, coherences present) and the mixture is
mixed (purity $\tfrac12$); they differ measurably in $\langle\sigma_x\rangle$.

In [ ]:
validate.close(
    np.array([purity(rho_sup), purity(rho_mix)]),
    np.array([1.0, 0.5]),
    "a superposition is pure (coherences present) and a 50/50 mixture is mixed",
    rtol=1e-9,
)
validate.check(
    abs(expectation(rho_sup, SIGMA_X) - 1.0) < 1e-9
    and abs(expectation(rho_mix, SIGMA_X)) < 1e-9,
    "the superposition gives ⟨σx⟩=1 (definite) while the mixture gives 0 (random)",
)

## Exercise 3 — Purity and the Bloch ball

Relate the purity to position in the Bloch ball: pure states on the surface, mixed states in the
interior, maximally mixed at the center.

1. Parametrize a qubit state by its Bloch vector $\mathbf r$: $\rho=\tfrac12(I+
   \mathbf r\cdot\boldsymbol\sigma)$.
2. Compute the purity as a function of the Bloch length $r=|\mathbf r|$.
3. Show purity $=1$ on the surface ($r=1$, pure) and $=\tfrac12$ at the center
   ($r=0$, maximally mixed).
4. Confirm the closed form $\mathrm{Tr}(\rho^2)=(1+r^2)/2$.

Cite {eq}`eq-purity`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

The qubit purity must follow $(1+r^2)/2$: $\tfrac12$ at the center (maximally
mixed) and $1$ on the Bloch surface (pure).

In [ ]:
validate.close(
    pur,
    (1 + r_lengths**2) / 2,
    "qubit purity is (1+r²)/2 — pure on the Bloch surface, maximally mixed at the center",
    rtol=1e-9,
)

## Exercise 4 — The reduced density matrix

Compute the state of one subsystem of a two-qubit system by the partial trace, and confirm it
reproduces local measurement statistics.

1. Take a two-qubit product state $|\psi\rangle_A\otimes|\phi\rangle_B$.
2. Implement the partial trace over $B$ (`numpy.reshape` to $(2,2,2,2)$ +
   `numpy.trace` of the $B$ indices).
3. Verify $\rho_A$ is a valid density matrix (Hermitian, positive, unit-trace).
4. Confirm $\mathrm{Tr}(\rho_A A)=\mathrm{Tr}(\rho_{AB}\,A\otimes I)$ for a local
   observable — the reduced state gives the right local statistics.

Cite {eq}`eq-partial-trace`.

In [ ]:
# (solution hidden on the public site)


### Validation 4

The partial trace must yield a valid density matrix for $A$ that reproduces local
expectation values: $\mathrm{Tr}(\rho_A\,\sigma_z)=\mathrm{Tr}(\rho_{AB}\,\sigma_z
\otimes I)$.

In [ ]:
validate.close(
    local,
    joint,
    "the reduced density matrix ρ_A = Tr_B ρ reproduces local expectation values",
    rtol=1e-9,
)
validate.check(
    np.isclose(np.trace(rho_A).real, 1.0)
    and np.all(np.linalg.eigvalsh(rho_A) > -1e-12),
    "the partial trace yields a valid density matrix (positive, unit-trace)",
)

## Exercise 5 — A pure whole with mixed parts: entanglement entropy

Show that a Bell state is pure but its subsystems are maximally mixed, and quantify with the von
Neumann entropy — the deep fact behind the random Bell marginals of [§6.8](bloch-sphere-entanglement.ipynb) and [§6.25](bell-inequality.ipynb).

1. Confirm the Bell state $(|00\rangle+|11\rangle)/\sqrt2$ has purity $1$ (a pure
   whole).
2. Compute $\rho_A=\mathrm{Tr}_B$ and show it is maximally mixed ($\rho_A=\tfrac12
   I$, purity $\tfrac12$).
3. Compute $S(\rho_A)=-\mathrm{Tr}(\rho_A\log_2\rho_A)=1$ bit.
4. Connect to [§6.8](bloch-sphere-entanglement.ipynb) / [§6.25](bell-inequality.ipynb): tracing out the partner leaves no coherence, so each
   particle alone looks random.

Cite {eq}`eq-entanglement-mixed`.

In [ ]:
# (solution hidden on the public site)


### Validation 5

A pure Bell state must have a maximally mixed reduced state: purity $\tfrac12$ and
von Neumann entropy $1$ bit.

In [ ]:
validate.close(
    np.array([purity(rho_bell_A), von_neumann_entropy(rho_bell_A)]),
    np.array([0.5, 1.0]),
    "a pure Bell state has a maximally mixed reduced state, S = 1 bit",
    rtol=1e-6,
)

## Exercise 6 — Entanglement across a family

Vary the entanglement of a two-qubit state continuously and track the entanglement entropy from
$0$ (product) to $1$ bit (Bell).

1. Build the family $\cos t\,|00\rangle+\sin t\,|11\rangle$.
2. For each $t$ compute $\rho_A=\mathrm{Tr}_B$ and $S(\rho_A)$.
3. Show $S$ runs from $0$ at $t=0$ (product state) to $1$ bit at $t=\pi/4$ (Bell).
4. Note $S(\rho_A)$ is the entanglement measure — maximal exactly when the state
   is maximally entangled.

Cite {eq}`eq-entanglement-mixed`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

The entanglement entropy must run from $0$ (product state, $t=0$) to $1$ bit (Bell
state, $t=\pi/4$).

In [ ]:
S_t0 = S_family[0]
S_tq = von_neumann_entropy(
    partial_trace(
        density_matrix((np.kron(KET0, KET0) + np.kron(KET1, KET1)) / np.sqrt(2)),
        0,
        (2, 2),
    )
)
validate.close(
    np.array([S_t0, S_tq]),
    np.array([0.0, 1.0]),
    "the entanglement entropy runs from 0 (product) to 1 bit (Bell)",
    atol=1e-6,
)

## Exercise 7 — Decoherence

Show that a superposition becomes a classical mixture when it entangles with an environment that
is then traced out — the quantum-to-classical mechanism.

1. Entangle a system superposition with an environment that records its state:
   $(|0\rangle|E_0\rangle+|1\rangle|E_1\rangle)/\sqrt2$.
2. Trace out the environment.
3. Show the system's off-diagonal coherence is proportional to the overlap
   $\langle E_0|E_1\rangle$.
4. As $\langle E_0|E_1\rangle\to0$ (the environment distinguishes the states),
   watch the coherence vanish and the purity drop to $\tfrac12$ — superposition
   into mixture.

Cite {eq}`eq-decoherence`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

As the environment overlap goes to zero, the system off-diagonal coherence must
decay to $0$ and the purity drop to $\tfrac12$: tracing out an entangled
environment decoheres a superposition into a mixture.

In [ ]:
validate.check(
    coh[0] > 0.49 and coh[-1] < 1e-9 and abs(pur_dec[-1] - 0.5) < 1e-9,
    "tracing out an entangled environment decoheres a superposition (off-diagonal "
    "0.5→0, purity 1→½) into a classical mixture",
)

## Exercise 8 — The von Neumann equation and conservation of purity *(student)*

Evolve a density matrix unitarily, confirm it satisfies the von Neumann equation, and show that
purity is conserved — closed dynamics never mixes.

1. Evolve $\rho(t)=U(t)\rho(0)U^\dagger(t)$ with $U=e^{-iHt/\hbar}$
   (`scipy.linalg.expm`, via `evolve_density`).
2. Confirm the von Neumann equation $i\hbar\,\mathrm d\rho/\mathrm dt=[H,\rho]$
   numerically (finite-difference the evolved $\rho$ against the commutator).
3. Show the purity stays constant — a pure state stays pure.
4. Contrast with decoherence (Ex. 7), where tracing out an environment *does*
   reduce purity. Closed dynamics never mixes; only the environment does.

Cite {eq}`eq-von-neumann`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

Unitary evolution must conserve purity (a pure state stays pure) and satisfy the
von Neumann equation; mixing requires an environment, not closed dynamics.

In [ ]:
validate.close(
    pur_unitary,
    np.ones_like(pur_unitary),
    "unitary (von Neumann) evolution conserves purity; mixing requires an environment",
    rtol=1e-6,
)
validate.check(
    residual < 1e-6, "the evolved ρ satisfies the von Neumann equation iℏ dρ/dt = [H,ρ]"
)

## Exercise 9 — (Synthesis) The state, honestly described

The wavefunction was always a best-case description — the state of a system we
know as well as nature permits. The density matrix is the honest generalization:
it describes what we actually have, whether that is a mixture we are classically
uncertain about or a fragment of an entangled whole whose partner we cannot see.
It draws the sharp line between a superposition and a mixture that a list of
probabilities would erase; it measures entanglement as the entropy of a part; and
it shows how a quantum system, by leaking its coherence into an environment it no
longer tracks, comes to look classical.

That last point deserves the emphasis this movement has been building toward. The
off-diagonal elements of the density matrix are where quantum mechanics lives:
keep them and you have interference, coherence, a working qubit; lose them to the
environment and you have an ordinary classical alternative. Decoherence is not
mysterious — it is bookkeeping about a partner you stopped following. And it
closed the loop on the whole movement: the random single-particle marginals of the
Bell state ([§6.8](bloch-sphere-entanglement.ipynb)) and the Bell test ([§6.25](bell-inequality.ipynb)) are now explained exactly, as the
maximally mixed reduced state of a pure entangled whole.

The same object, carrying a thermal weight $e^{-\beta H}$, will later be the
starting point of quantum statistical mechanics — but that is Volume VII. Here it
has told us what the entangled states of this movement are truly made of. The
final notebook, an optional capstone ([§6.27](quantum-information.ipynb)), takes these ideas — mixed states,
entropy, entanglement — into the language of quantum information.

## Notebook summary

- **The density operator** {eq}`eq-density-operator`: $\rho=|\psi\rangle\langle
  \psi|$ (`numpy.outer`) or $\sum_i p_i|\psi_i\rangle\langle\psi_i|$; Hermitian,
  positive, unit-trace, with $\langle A\rangle=\mathrm{Tr}(\rho A)$.
- **Superposition vs mixture** {eq}`eq-superposition-mixture`: same diagonal,
  different coherences; purity $1$ vs $\tfrac12$; $\langle\sigma_x\rangle$ $1$ vs $0$.
- **Purity** {eq}`eq-purity`: $\mathrm{Tr}(\rho^2)=(1+r^2)/2$ — the Bloch sphere
  filled in, pure on the surface and maximally mixed at the center.
- **The partial trace** {eq}`eq-partial-trace`: $\rho_A=\mathrm{Tr}_B\rho$
  (`numpy.reshape` + `numpy.trace`) gives correct local statistics.
- **Entanglement = mixed parts** {eq}`eq-entanglement-mixed`: a pure Bell state has
  $\rho_A=\tfrac12 I$, $S=1$ bit — *why* [§6.8](bloch-sphere-entanglement.ipynb)/[§6.25](bell-inequality.ipynb) marginals are random; $S(\rho_A)$
  runs $0\to1$ bit across the entanglement family.
- **Decoherence** {eq}`eq-decoherence`: tracing out an environment sends the
  coherence $\to0$ and purity $\to\tfrac12$ — superposition into mixture.
- **Von Neumann evolution** {eq}`eq-von-neumann`: $i\hbar\dot\rho=[H,\rho]$; closed
  dynamics conserves purity (`scipy.linalg.expm`), so mixing needs an environment.

> **Scope reminder.** The thermal density matrix $\rho=e^{-\beta H}/Z$ and quantum
> statistical mechanics are built on exactly this object — but that is Volume VII.
> Here $\rho$ served entanglement, mixed states, and decoherence.

## Outlook

- **Quantum information** ([§6.27](quantum-information.ipynb), optional capstone): density matrices, entropy, and
  quantum channels.
- **The thermal density matrix** $\rho=e^{-\beta H}/Z$ and quantum statistical
  mechanics — the deferred home (Volume VII).
- **Open quantum systems and Lindblad dynamics**; the full theory of decoherence
  (horizons).
- **Quantum error correction** and why decoherence is the central obstacle to
  quantum computing (a horizon).
- Cross-reference [§6.8](bloch-sphere-entanglement.ipynb) (Bloch sphere, Bell reduced state), [§6.25](bell-inequality.ipynb) (Bell marginals),
  [§6.5](postulates.ipynb) (Born rule), [§6.7](time-evolution.ipynb) (unitary evolution), forward to [§6.27](quantum-information.ipynb), Volume VII.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()